# GenAI Workflow and Architecture Patterns (Capstone)

## 1. End-to-End Product Lens
A production GenAI system is an iterative lifecycle: problem definition, model strategy, prompting/RAG/fine-tuning decisions, evaluation, deployment, and monitoring with feedback loops. Interviewers expect candidates to think beyond model calls and address reliability, cost, and governance. The strongest answers describe decisions as evidence-driven and revisited over time, not fixed at launch.

## 2. Problem Framing and Success Metrics
Start by defining user outcomes, constraints, and measurable success criteria. Typical metrics include task success, factual precision, latency, cost per request, and human override frequency. A high-signal interview answer states that architecture should be chosen from business objectives first, then technical preferences.

## 3. Model Selection in Context
Model selection is a multi-objective optimization problem across quality, context window, safety behavior, throughput, and price. Bigger models can reduce prompt complexity but increase serving cost and latency. Interview-ready framing: benchmark on your own workload slices and compare variance under real prompt distributions, not just leaderboard averages.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

models = ['Small-LLM', 'Mid-LLM', 'Large-LLM']
quality = np.array([0.71, 0.82, 0.88])
latency = np.array([220, 420, 760])
cost = np.array([1.0, 2.0, 4.4])

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].bar(models, quality, color='tab:blue')
ax[0].set_ylim(0, 1)
ax[0].set_title('Quality comparison')

ax[1].plot(models, latency, marker='o', label='Latency (ms)')
ax[1].plot(models, cost*150, marker='s', label='Scaled cost')
ax[1].set_title('Latency and cost trend')
ax[1].legend()

plt.tight_layout()
plt.show()

## 4. Prompting vs RAG vs Fine-Tuning
Prompting is ideal for rapid iteration when base model capability is already close to target behavior. RAG adds external grounding for changing proprietary knowledge, while fine-tuning helps enforce domain style or repetitive patterns at scale. In interviews, explicitly discuss trigger conditions for each approach and mention that hybrid stacks are common in mature systems.

## 5. Architecture Patterns Comparison Table
| Pattern | Best Use Case | Strengths | Weaknesses | Typical Risks |
|---|---|---|---|---|
| Prompt-only | Low-complexity assistant tasks | Fast to ship, low infra overhead | Weak grounding on private knowledge | Hallucinated specifics |
| RAG | Enterprise knowledge QA | Better factual grounding, fresher data | Retrieval quality dependency | Wrong chunk retrieval |
| Agent + Tools | Multi-step workflows and actions | Dynamic planning and execution | More operational complexity | Tool misuse, loops |
| Fine-tuned model | High-volume domain style tasks | Consistency and lower prompt burden | Data curation and retrain overhead | Drift after policy changes |

In [ ]:
def architecture_selector(task_complexity, private_knowledge, needs_actions, style_consistency):
    if task_complexity == 'low' and not private_knowledge and not needs_actions:
        return 'prompt_only'
    if needs_actions:
        return 'agent_with_tools'
    if private_knowledge:
        return 'rag'
    if style_consistency:
        return 'fine_tuned'
    return 'prompt_only'

cases = [
    ('low', False, False, False),
    ('medium', True, False, False),
    ('high', True, True, False),
    ('medium', False, False, True)
]

for c in cases:
    print(c, '->', architecture_selector(*c))

## 6. Retrieval and Knowledge Strategy
RAG quality depends on chunking, embeddings, metadata filters, and ranking logic. Many teams over-focus on model size while under-investing in retrieval hygiene and source authority scoring. Interview tip: propose a retrieval evaluation suite with answer support checks, citation precision, and failure clustering by document type.

## 7. Fine-Tuning Decision Criteria
Fine-tuning is justified when repetitive domain behavior cannot be reliably achieved through prompting and retrieval alone, especially at high request volume. It can reduce token costs and increase style consistency, but demands versioning discipline and retraining governance. A strong interview answer includes data governance, bias checks, and rollback strategy.

In [ ]:
from dataclasses import dataclass

@dataclass
class WorkflowChecklist:
    problem_defined: bool
    metrics_defined: bool
    baseline_tested: bool
    architecture_selected: bool
    safety_guardrails_ready: bool
    eval_suite_ready: bool
    deployment_plan_ready: bool
    monitoring_plan_ready: bool

def readiness_score(item: WorkflowChecklist) -> float:
    vals = list(item.__dict__.values())
    return 100.0 * sum(vals) / len(vals)

sample = WorkflowChecklist(True, True, True, True, False, True, False, False)
print(f'Workflow readiness score: {readiness_score(sample):.1f}%')

## 8. Evaluation Framework and Release Gates
Evaluation should include offline benchmark sets, adversarial tests, human review slices, and online shadow evaluation before full rollout. Define hard release gates for factuality, policy compliance, and latency so launch decisions are objective. Interviewers appreciate when candidates differentiate acceptance thresholds for low-risk vs high-impact use cases.

## 9. Deployment Patterns
Common deployment patterns include canary rollouts, feature flags, and fallback to deterministic responses when model confidence is low. Rollout strategy should include safe rollback paths and incident ownership. In interviews, mention that deployment is part of architecture quality because operational design determines how safely you learn from production.

In [ ]:
weeks = np.arange(1, 9)
quality = np.array([0.65, 0.69, 0.72, 0.76, 0.79, 0.81, 0.83, 0.84])
latency = np.array([740, 700, 660, 620, 590, 560, 550, 540])

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(weeks, quality, marker='o', color='tab:blue')
ax1.set_xlabel('Week')
ax1.set_ylabel('Quality score', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.plot(weeks, latency, marker='s', color='tab:orange')
ax2.set_ylabel('Latency (ms)', color='tab:orange')
ax2.tick_params(axis='y', labelcolor='tab:orange')

plt.title('Post-launch trend: quality up, latency down')
fig.tight_layout()
plt.show()

## 10. Monitoring and Feedback Loops
Monitoring should track quality drift, hallucination incidents, retrieval miss rate, safety violations, and user correction patterns. Feedback loops convert observed failures into prompt updates, retrieval improvements, and policy tuning. Interview insight: monitoring is not passive dashboards; it is an active mechanism for model and architecture evolution.

## 11. Cost and Capacity Planning
Capacity planning should estimate token usage, tool calls, peak concurrency, and storage growth for logs and embeddings. Cost-aware design often includes request routing, caching, and selective use of larger models only when needed. In interviews, demonstrate practical thinking by proposing cost budgets linked to measurable quality targets.

In [ ]:
architectures = ['Prompt', 'RAG', 'Agent', 'Fine-tuned+RAG']
monthly_cost = np.array([1200, 2200, 3400, 3000])
business_value = np.array([45, 68, 82, 86])

roi = business_value / monthly_cost
for a, c, v, r in zip(architectures, monthly_cost, business_value, roi):
    print(f'{a:15} cost={c:4.0f} value={v:3.0f} ROI={r:.4f}')

plt.figure(figsize=(7.2, 4))
plt.bar(architectures, roi)
plt.ylabel('Value per cost unit')
plt.title('Illustrative ROI comparison by architecture')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 12. Decision Flowchart in Markdown
Use this flowchart during architecture reviews:

Start -> Is the task simple and low-risk?
  -> Yes: Prompt-only baseline
  -> No: Does response require private or frequently changing knowledge?
      -> Yes: RAG baseline
      -> No: Does the system need external actions and multi-step planning?
          -> Yes: Agent with tools
          -> No: Is strict domain style needed at scale?
              -> Yes: Fine-tuning (possibly with RAG)
              -> No: Prompt-only with strong eval loop

Always re-evaluate after monitoring signals and product changes.

## 13. Governance and Responsible Deployment
Workflow excellence includes governance artifacts: model cards, risk assessments, evaluation reports, and incident procedures. Architecture is not complete without accountability and compliance readiness. Interviewers notice when candidates explicitly include legal/security collaboration and auditable controls as part of system design.

## 14. Interview Framework: Designing GenAI for X
For "Design a GenAI system for X" questions, use a repeatable structure: define users and risk, choose architecture baseline, detail data and retrieval strategy, set metrics, define guardrails, and plan staged rollout. This demonstrates senior-level product and engineering judgment. Keep trade-offs explicit so interviewers can see your reasoning process.

## 15. Interview Q&A (9 Detailed Questions)
**Q1: How do you design a GenAI system for a domain task?** A1: Start with problem and constraints, choose architecture baseline, define eval gates, then deploy incrementally with monitoring.

**Q2: Prompt vs RAG vs fine-tune: how do you choose?** A2: Prompt for speed, RAG for evolving private knowledge, fine-tune for repeated specialized behavior.

**Q3: When should you introduce an agent pattern?** A3: When workflows require dynamic multi-step decisions and external actions.

**Q4: What is the biggest RAG failure mode?** A4: Retrieval mismatch that grounds answers in irrelevant context.

**Q5: What are must-have evaluation metrics?** A5: Task success, factual support, latency, cost, and policy compliance.

**Q6: Why are release gates important?** A6: They enforce objective launch readiness and reduce risky subjective decisions.

**Q7: What should monitoring include post-launch?** A7: Drift, safety incidents, retrieval quality, user corrections, and cost anomalies.

**Q8: How do you justify fine-tuning cost?** A8: Show measurable gains in consistency, throughput, or reduced prompt/token overhead.

**Q9: What impresses interviewers most?** A9: An end-to-end design that balances quality, reliability, cost, and governance.

## 16. Summary and Key Interview Takeaways
Capstone workflow thinking turns GenAI from a demo into an operating system for product value. The right architecture is context-dependent and should evolve based on measured outcomes.

**Key Takeaways:**
- Start from user outcomes and risk level, not model hype.
- Use prompt, RAG, agent, and fine-tune patterns as tools, not dogma.
- Enforce evaluation gates before deployment.
- Build monitoring loops that drive continuous improvement.
- In interviews, present architecture decisions with explicit trade-offs.